# Training pipeline: allocating undecided 2023 PSOE voters

**Business problem** (full framing in `REPORT.md`): a polling firm publishes a monthly vote estimate. Among
the people who voted PSOE in July 2023, about one in seven says they don't know yet what they will vote, or
won't say. The estimate has to assign them: back to PSOE, or not. The model is trained on the 2023 PSOE
voters who *did* declare an intention and is used to allocate the ones who didn't.

That framing drives the pipeline:

| Decision | Choice | Why |
|---|---|---|
| Target | `churn = 1` if the declared intention is anything but PSOE (another party, blank, null, not voting) | what the estimate needs to know is who goes back to PSOE |
| Rows | trained and evaluated on declared voters only; the undecided are what the model allocates | the undecided have no label |
| Model | Logistic regression | a probability per voter: thresholded into an assignment, or averaged into an expected share |
| Split | By barometer month: train Apr 2025–Jan 2026, validate Feb–Apr 2026, test May–Sep 2026 | each month's estimate is made with a model fitted on past months; a random split would leak month-level shocks (July 2025: 31.4% churn among declared voters, against 21.1% in June) into training |
| Threshold | Minimises the monthly error of the estimated churn rate on the validation months, in percentage points | the pollster's loss is a wrong published number, not a wrong individual |
| Evaluation | Declared voters in the test months are masked and allocated as if they were undecided | the only place where the truth is known |
| Weights | Each barometer re-weighted on its whole sample (recall of the 23J vote + sex × age + education, the GAD3 observatory / TFG correction, `data/raking.py`); every population figure uses it, the model fit does not | the CIS sample over-represents 2023 PSOE voters, and a published estimate is a population figure |
| Tuning | `C` (L2 strength) and, for PyTorch, the learning rate; nothing else | effort goes into features (brief, step 5) |

Run via `uv run python main.py train`. Writes everything the dashboard needs to `artifacts/`.

## Setup

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score, silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import features as F
from models import DTYPE, LogisticNet, manual_pytorch_proba, monthly_abs_error, monthly_errors, nn_module_proba

ROOT = Path.cwd()  # main.py runs this with cwd = repo root
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

C_GRID = [0.001, 0.01, 0.1, 1.0, 10.0]
LR_GRID = [0.1, 0.5, 1.0, 2.0, 4.0]
EPOCHS = 10_000
MIN_LOGLOSS_GAIN = 0.0005     # a feature block must earn at least this on validation
K_CANDIDATES = range(3, 9)

## Load, engineer, split

`features.engineer` is pure (nothing fitted), so it can run on the whole file before
splitting without leaking anything; everything *fitted* (imputation medians, scaling,
one-hot categories, k-means) is fitted on the training months only, below.

`weight` is the corrected survey weight built in `data/build_dataset.py` (see `data/raking.py`). It is used for
every figure that describes the population (actual and estimated churn, the pollster's error, the allocation),
not to fit the models: sex, age and education, the demographic axes of the correction, are already features.

In [2]:
raw = F.load_raw()
eng = F.engineer(raw)
eng["split"] = F.split_of(raw["study"])
eng[F.TARGET] = raw[F.TARGET]
declared = F.is_declared(raw)
undecided = ~declared

tr, va, te = ((eng["split"] == s) & declared for s in ("train", "val", "test"))
y_train, y_val, y_test = (eng.loc[m, F.TARGET].to_numpy() for m in (tr, va, te))
month_val, month_test = (raw.loc[m, "study"].to_numpy() for m in (va, te))
w_train, w_val, w_test = (raw.loc[m, "weight"].to_numpy() for m in (tr, va, te))  # corrected weights

print(f"declared: {declared.sum():,} rows | undecided, to allocate: {undecided.sum():,} rows")
print(eng[declared].groupby("split")[F.TARGET].agg(rows="size", churn_rate="mean").round(3))
print("\nBy month (the July 2025 spike is study 3517):")
def wmean_by_study(values: pd.Series, rows: pd.Series) -> pd.Series:
    w = raw.loc[rows, "weight"]
    return (values[rows] * w).groupby(raw.loc[rows, "study"]).sum() / w.groupby(raw.loc[rows, "study"]).sum()


print(pd.DataFrame({
    "undecided_share": wmean_by_study(undecided.astype(float), undecided | declared),
    "churn_among_declared": wmean_by_study(eng[F.TARGET].astype(float), declared),
    "churn_unweighted": eng.loc[declared, F.TARGET].groupby(raw.loc[declared, "study"]).mean(),
}).round(3).to_string())

declared: 15,994 rows | undecided, to allocate: 2,509 rows
       rows  churn_rate
split                  
test   4172       0.249
train  8741       0.237
val    3081       0.245

By month (the July 2025 spike is study 3517):
       undecided_share  churn_among_declared
study                                       
3505             0.137                 0.196
3510             0.134                 0.171
3514             0.136                 0.211
3517             0.177                 0.314
3524             0.150                 0.244
3528             0.132                 0.252
3530             0.138                 0.232
3536             0.142                 0.259
3540             0.158                 0.247
3544             0.127                 0.260
3546             0.132                 0.267
3557             0.109                 0.212
3562             0.114                 0.206
3567             0.152                 0.262
3571             0.115                 0.268
3577     

## Voter segments (unsupervised): k-means on the attitude block

Fitted on the **training months only**. Inputs are the political attitudes (four leader
ratings, ideology, turnout intent, economy) so segments read as political profiles.
`k` is chosen by silhouette among 3–8, and the segment is a candidate feature block: the
ablation decides whether it earns its place.

In [3]:
def make_segmenter(k: int) -> Pipeline:
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("kmeans", KMeans(n_clusters=k, n_init=10, random_state=SEED)),
    ])


X_seg_train = eng.loc[tr, F.SEGMENT_INPUTS]
sample = X_seg_train.sample(4000, random_state=SEED).index
silhouettes = {}
for k in K_CANDIDATES:
    seg = make_segmenter(k).fit(X_seg_train)
    Z = seg[:-1].transform(X_seg_train.loc[sample])
    silhouettes[k] = float(silhouette_score(Z, seg[-1].predict(Z)))
best_k = max(silhouettes, key=silhouettes.get)
print("silhouette by k:", {k: round(s, 3) for k, s in silhouettes.items()}, "-> k =", best_k)

segmenter = make_segmenter(best_k).fit(X_seg_train)
eng["segment"] = "S" + pd.Series(segmenter.predict(eng[F.SEGMENT_INPUTS]), index=eng.index).astype(str)

profile = eng.loc[tr].groupby("segment").agg(
    size=(F.TARGET, "size"), churn=(F.TARGET, "mean"),
    **{c: (c, "mean") for c in F.SEGMENT_INPUTS},
).round(2)
print(profile.to_string())

silhouette by k: {3: 0.279, 4: 0.21, 5: 0.192, 6: 0.177, 7: 0.181, 8: 0.177} -> k = 3
         size  churn  rate_sanchez  rate_feijoo  rate_diaz  rate_abascal  ideology  probvoto  eco_spain  eco_personal
segment                                                                                                              
S0        730   0.60          4.67         5.18       4.39          5.33      5.28      9.23       2.58          3.37
S1       6015   0.09          7.91         2.75       6.72          1.14      2.84      9.87       3.82          3.86
S2       1996   0.53          4.13         2.62       3.38          1.22      3.99      7.73       2.51          3.40


## Shared preprocessing

One `ColumnTransformer` feeds all three implementations, so they see the identical
design matrix. Numeric: median imputation + standardisation (needed for gradient
descent to converge at one learning rate, and for L2 to penalise every coefficient on
the same scale). Categorical: one-hot, with categories seen fewer than 50 times in
training pooled into an "infrequent" column (e.g. Ceuta and Melilla, ~40 voters each).

In [4]:
def build_preprocessor(blocks: list[str]) -> ColumnTransformer:
    num, binary, cat = F.block_columns(blocks)
    parts = []
    if num:
        parts.append(("numeric", Pipeline([("impute", SimpleImputer(strategy="median")),
                                           ("scale", StandardScaler())]), num))
    if binary:
        parts.append(("binary", "passthrough", binary))
    if cat:
        parts.append(("categorical", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                                   min_frequency=50, sparse_output=False), cat))
    return ColumnTransformer(parts)


def sklearn_val_scores(blocks: list[str], C: float) -> dict:
    pre = build_preprocessor(blocks).fit(eng.loc[tr])
    model = LogisticRegression(C=C, max_iter=10_000).fit(pre.transform(eng.loc[tr]), y_train)
    p = model.predict_proba(pre.transform(eng.loc[va]))[:, 1]
    return {"val_logloss": log_loss(y_val, p), "val_auc": roc_auc_score(y_val, p),
            "n_features": len(pre.get_feature_names_out())}

## Feature-block ablation (validation months)

Blocks are added one at a time in a fixed order (`features.BLOCKS`). A block is kept only
if it lowers validation log-loss by at least `MIN_LOGLOSS_GAIN`; the rest are reported as
negative results. Log-loss rather than AUC because the estimate is built from the
probabilities themselves (thresholded or averaged), not just from the ranking.

In [5]:
ablation = []
selected = ["core"]
base = sklearn_val_scores(selected, C=1.0)
ablation.append({"block": "core", **base, "gain": None, "kept": True})
for block in [b for b in F.BLOCKS if b != "core"]:
    scores = sklearn_val_scores(selected + [block], C=1.0)
    gain = base["val_logloss"] - scores["val_logloss"]
    kept = gain >= MIN_LOGLOSS_GAIN
    ablation.append({"block": block, **scores, "gain": gain, "kept": kept})
    if kept:
        selected.append(block)
        base = scores

print(pd.DataFrame(ablation).round(4).to_string(index=False))
print("\nselected blocks:", selected)

                 block  val_logloss  val_auc  n_features    gain  kept
                  core       0.3553   0.8701           8     NaN  True
       dont_know_flags       0.3554   0.8697          14 -0.0001 False
       competitive_gap       0.3416   0.8820          10  0.0137  True
         closest_party       0.3042   0.9141          17  0.0374  True
ideology_turnout_shape       0.3040   0.9146          24  0.0002 False
        problem_themes       0.3036   0.9141          27  0.0005  True
     sociodemographics       0.3021   0.9154          81  0.0015  True
               segment       0.3021   0.9157          84  0.0001 False

selected blocks: ['core', 'competitive_gap', 'closest_party', 'problem_themes', 'sociodemographics']


## How firm is the block selection?

The forward pass above depends on the order the blocks are tried in, and on a single validation sample.
Two checks on the **final** set of blocks:

- every kept block is **dropped** from the final set: does validation log-loss get worse without it?
- every rejected block is **added** to the final set: would it have earned its place if it had been tried last?

`gain > 0` means the block lowers validation log-loss. The interval is a paired bootstrap over validation
voters (1,000 resamples of the per-voter log-loss difference): an interval that crosses 0 means the block's
contribution can't be told apart from noise on 3.5k voters. Nothing is re-selected here; this only reports
how firm the selection is.

In [6]:
def val_proba(blocks: list[str], C: float = 1.0) -> np.ndarray:
    pre = build_preprocessor(blocks).fit(eng.loc[tr])
    model = LogisticRegression(C=C, max_iter=10_000).fit(pre.transform(eng.loc[tr]), y_train)
    return model.predict_proba(pre.transform(eng.loc[va]))[:, 1]


def row_logloss(y, p) -> np.ndarray:
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))


N_BOOT = 1000
rng = np.random.default_rng(SEED)
boot_val = rng.integers(0, len(y_val), size=(N_BOOT, len(y_val)))


def gain_with_ci(loss_without: np.ndarray, loss_with: np.ndarray) -> tuple[float, float, float]:
    d = loss_without - loss_with  # per voter; > 0 means the block helps
    boot = d[boot_val].mean(axis=1)
    return float(d.mean()), float(np.quantile(boot, 0.025)), float(np.quantile(boot, 0.975))


loss_final = row_logloss(y_val, val_proba(selected))
ablation_check = []
for block in [b for b in F.BLOCKS if b != "core"]:
    if block in selected:
        loss_other = row_logloss(y_val, val_proba([b for b in selected if b != block]))
        gain, lo, hi = gain_with_ci(loss_other, loss_final)
    else:
        loss_other = row_logloss(y_val, val_proba(selected + [block]))
        gain, lo, hi = gain_with_ci(loss_final, loss_other)
    ablation_check.append({"block": block, "in_final": block in selected,
                           "check": "dropped from final" if block in selected else "added to final",
                           "gain": gain, "ci_low": lo, "ci_high": hi, "ci_excludes_0": lo > 0 or hi < 0})
print(pd.DataFrame(ablation_check).round(4).to_string(index=False))

                 block  in_final              check    gain  ci_low  ci_high  ci_excludes_0
       dont_know_flags     False     added to final  0.0000 -0.0011   0.0012          False
       competitive_gap      True dropped from final  0.0076  0.0035   0.0116           True
         closest_party      True dropped from final  0.0318  0.0233   0.0408           True
ideology_turnout_shape     False     added to final  0.0001 -0.0024   0.0026          False
        problem_themes      True dropped from final -0.0000 -0.0022   0.0023          False
     sociodemographics      True dropped from final  0.0015 -0.0024   0.0054          False
               segment     False     added to final  0.0001 -0.0011   0.0011          False


## Tune regularisation strength `C` (sklearn, validation log-loss)

In [7]:
c_scores = {C: sklearn_val_scores(selected, C)["val_logloss"] for C in C_GRID}
best_C = min(c_scores, key=c_scores.get)
print({C: round(s, 4) for C, s in c_scores.items()}, "-> C =", best_C)

preprocessor = build_preprocessor(selected).fit(eng.loc[tr])
feature_names = list(preprocessor.get_feature_names_out())
X_train, X_val, X_test = (preprocessor.transform(eng.loc[m]) for m in (tr, va, te))
n_train = X_train.shape[0]

# sklearn minimises  C * sum(logloss) + ||w||^2 / 2.  Dividing by C*n gives the
# mean-loss form the PyTorch loops use: mean(logloss) + lambda * ||w||^2 with
lambda_reg = 1.0 / (2 * best_C * n_train)
print(f"{len(feature_names)} features, lambda = {lambda_reg:.2e}")

{0.001: 0.3412, 0.01: 0.3033, 0.1: 0.3004, 1.0: 0.3021, 10.0: 0.3026} -> C = 0.1
81 features, lambda = 5.72e-04


## Model 1 of 3: scikit-learn `LogisticRegression`

L-BFGS, L2 penalty, intercept not penalised. Tight tolerance so it is a trustworthy
reference optimum for the two gradient-descent versions.

In [8]:
sk_model = LogisticRegression(C=best_C, max_iter=10_000, tol=1e-10).fit(X_train, y_train)

## Model 2 of 3: from-scratch PyTorch loop

Raw tensors, autograd for the gradient, a manual `w -= lr * grad` update: the
Session 5 approach. Full-batch gradient descent on
`mean BCE + lambda * ||w||^2` (bias unpenalised, as in sklearn).

In [9]:
def manual_pytorch_fit(X, y, lr: float, lam: float, epochs: int):
    X_t = torch.tensor(X, dtype=DTYPE)
    y_t = torch.tensor(y, dtype=DTYPE).view(-1, 1)
    w = torch.zeros(X_t.shape[1], 1, dtype=DTYPE, requires_grad=True)
    b = torch.zeros(1, dtype=DTYPE, requires_grad=True)
    history = []
    for epoch in range(epochs):
        z = X_t @ w + b
        p = torch.sigmoid(z)
        bce = -torch.mean(y_t * torch.log(p + 1e-12) + (1 - y_t) * torch.log(1 - p + 1e-12))
        loss = bce + lam * torch.sum(w ** 2)
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad
            b -= lr * b.grad
            w.grad.zero_()
            b.grad.zero_()
        if epoch % 50 == 0:
            history.append(loss.item())
    return w.detach(), b.detach(), history

## Model 3 of 3: standard `torch.nn.Module` + `torch.optim`

Same data, same objective: `BCEWithLogitsLoss` (numerically stable sigmoid + BCE) and
plain SGD with `weight_decay = 2 * lambda` on the weight only (the gradient of
`lambda * ||w||^2` is `2 * lambda * w`) and no decay on the bias.

In [10]:
def nn_module_fit(X, y, lr: float, lam: float, epochs: int):
    X_t = torch.tensor(X, dtype=DTYPE)
    y_t = torch.tensor(y, dtype=DTYPE).view(-1, 1)
    model = LogisticNet(X_t.shape[1])
    with torch.no_grad():
        model.linear.weight.zero_()
        model.linear.bias.zero_()
    optimizer = torch.optim.SGD([
        {"params": [model.linear.weight], "weight_decay": 2 * lam},
        {"params": [model.linear.bias], "weight_decay": 0.0},
    ], lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    history = []
    for epoch in range(epochs):
        optimizer.zero_grad()
        loss = criterion(model(X_t), y_t)
        loss.backward()
        optimizer.step()
        if epoch % 50 == 0:
            history.append((loss + lam * model.linear.weight.pow(2).sum()).item())
    return model, history

## Tune the learning rate (training objective, not validation)

The problem is convex: `C` fixes *where* the optimum is, and the learning rate only
decides whether `EPOCHS` steps of gradient descent *reach* it (too small: stops short;
too large: overshoots and oscillates). So the learning rate is chosen by the training
objective it reaches, not by validation log-loss: choosing on validation would reward a
run that stops short, which is early stopping acting as hidden extra regularisation on
top of `C`. Theory check: plain gradient descent is stable for `lr < 2 / L`, with
`L <= 0.25 * lambda_max(X'X / n) + 2 * lambda`.

In [11]:
def objective(X, y, w: np.ndarray, b: float) -> float:
    p = 1 / (1 + np.exp(-(X @ w + b)))
    return log_loss(y, p) + lambda_reg * float(np.sum(w ** 2))


sk_objective = objective(X_train, y_train, sk_model.coef_.ravel(), sk_model.intercept_[0])
L_bound = 0.25 * np.linalg.eigvalsh(X_train.T @ X_train / n_train).max() + 2 * lambda_reg
print(f"sklearn objective {sk_objective:.10f} | GD stable for lr < {2 / L_bound:.2f} (conservative)")

lr_objective = {}
for lr in LR_GRID:
    w, b, _ = manual_pytorch_fit(X_train, y_train, lr, lambda_reg, EPOCHS)
    obj = objective(X_train, y_train, w.numpy().ravel(), b.item())
    lr_objective[lr] = obj if np.isfinite(obj) else np.inf
    print(f"lr={lr:<4} objective gap to sklearn = {lr_objective[lr] - sk_objective:.2e}")
best_lr = min(lr_objective, key=lr_objective.get)
print("-> lr =", best_lr)

sklearn objective 0.2879522551 | GD stable for lr < 1.66 (conservative)


lr=0.1  objective gap to sklearn = 1.43e-05


lr=0.5  objective gap to sklearn = 3.32e-07


lr=1.0  objective gap to sklearn = 5.45e-09


lr=2.0  objective gap to sklearn = 1.44e-12


lr=4.0  objective gap to sklearn = 1.30e-02
-> lr = 2.0


## Final fit of the PyTorch models on the training months

In [12]:
w_manual, b_manual, hist_manual = manual_pytorch_fit(X_train, y_train, best_lr, lambda_reg, EPOCHS)
nn_model, hist_nn = nn_module_fit(X_train, y_train, best_lr, lambda_reg, EPOCHS)
X_undecided = preprocessor.transform(eng.loc[undecided])

proba = {}
for split, X in (("val", X_val), ("test", X_test), ("undecided", X_undecided)):
    proba[split] = {
        "sklearn": sk_model.predict_proba(X)[:, 1],
        "manual_pytorch": manual_pytorch_proba(X, w_manual, b_manual),
        "nn_module": nn_module_proba(X, nn_model),
    }

## Decision threshold: the error of the published estimate

Each masked voter is assigned `churn` when `p > t`. A month's estimated churn rate is the share assigned, and
the pollster's error is *estimated − actual*, in percentage points. `t` minimises the mean absolute monthly
error on the **validation** months; the test months are only used to report.

The model's probabilities also give an estimate without any threshold (the mean of `p` in each month), reported
alongside as a reference.

Baselines, scored the same way:
- **all return**: every undecided voter goes back to PSOE (estimated churn 0), a common rule of thumb;
- **naive constant**: every month gets the training churn rate;
- **closest-party rule**: back to PSOE if the party they feel closest to is PSOE, the logic of the CIS's own
  vote + sympathy estimate.

In [13]:
THRESHOLDS = np.round(np.arange(0.01, 1.0, 0.01), 2)

best_t = {}
for name, p in proba["val"].items():
    errors_val = [monthly_abs_error(y_val, p > t, month_val, w_val) for t in THRESHOLDS]
    best_t[name] = float(THRESHOLDS[int(np.argmin(errors_val))])
print("thresholds chosen on validation:", best_t)

train_rate = float(np.average(y_train, weights=w_train))
rule_churn = lambda m: (eng.loc[m, "closest_group"] != "psoe").to_numpy()  # noqa: E731

thresholds chosen on validation: {'sklearn': 0.32, 'manual_pytorch': 0.32, 'nn_module': 0.32}


### Why is the threshold where it is?

The loss only looks at the count of voters assigned to each side. Compare the chosen threshold with 0.5, the
default a classifier would use: false positives vs false negatives, accuracy, and the monthly error. Calibration
by decile matters too: if the probabilities are off, the threshold has to compensate.

In [14]:
p_val = proba["val"]["sklearn"]
val_deciles = (pd.DataFrame({"p": p_val, "y": y_val})
               .groupby(pd.qcut(p_val, 10), observed=True)
               .agg(mean_pred=("p", "mean"), actual=("y", "mean"), n=("y", "size")))
print(val_deciles.round(3).to_string())

threshold_check = {"val_deciles": val_deciles.reset_index(drop=True).to_dict(orient="records"), "by_threshold": []}
for t in (best_t["sklearn"], 0.5):
    a = p_val > t
    row = {"threshold": t, "false_pos": int((a & (y_val == 0)).sum()), "false_neg": int((~a & (y_val == 1)).sum()),
           "accuracy": float((a == (y_val == 1)).mean()), "monthly_abs_error_pp": monthly_abs_error(y_val, a, month_val, w_val)}
    threshold_check["by_threshold"].append(row)
print()
print(pd.DataFrame(threshold_check["by_threshold"]).round(3).to_string(index=False))
threshold_check["expected_val_monthly_abs_error_pp"] = monthly_abs_error(y_val, p_val, month_val, w_val)
print("no threshold (mean of p), validation:", round(threshold_check["expected_val_monthly_abs_error_pp"], 3), "pp")

                                  mean_pred  actual    n
(0.0010500000000000002, 0.00881]      0.006   0.013  309
(0.00881, 0.0161]                     0.012   0.003  308
(0.0161, 0.0276]                      0.021   0.016  308
(0.0276, 0.0421]                      0.034   0.042  308
(0.0421, 0.0691]                      0.054   0.071  308
(0.0691, 0.119]                       0.092   0.101  308
(0.119, 0.205]                        0.157   0.253  308
(0.205, 0.398]                        0.292   0.364  308
(0.398, 0.826]                        0.583   0.627  308
(0.826, 0.999]                        0.944   0.958  308

 threshold  false_pos  false_neg  accuracy  monthly_abs_error_pp
      0.32        191        223     0.866                 1.412
      0.50         76        318     0.872                 7.929


## Three-way comparison on the test months

In [15]:
def scores(y, month, w, p=None, assign=None, estimate=None) -> dict:
    """`assign`: 0/1 per voter (thresholded). `estimate`: a probability per voter (expected value).
    Population figures (churn rates, errors) use the corrected weight `w`; AUC, log-loss, Brier and the
    confusion counts are about individuals and are not weighted."""
    est = assign.astype(float) if assign is not None else estimate
    out = {
        "auc": roc_auc_score(y, p) if p is not None else None,
        "logloss": log_loss(y, p) if p is not None else None,
        "brier": brier_score_loss(y, p) if p is not None else None,
        "estimated_churn": float(np.average(est, weights=w)), "actual_churn": float(np.average(y, weights=w)),
        "error_pp": 100 * float(np.average(est, weights=w) - np.average(y, weights=w)),
        "monthly_abs_error_pp": monthly_abs_error(y, est, month, w),
        "false_pos": None, "false_neg": None, "accuracy": None,
    }
    if assign is not None:
        out |= {"false_pos": int((assign & (y == 0)).sum()), "false_neg": int((~assign & (y == 1)).sum()),
                "accuracy": float((assign == (y == 1)).mean())}
    return out


results = {}
for name, p in proba["test"].items():
    results[name] = {"threshold": best_t[name], **scores(y_test, month_test, w_test, p=p, assign=p > best_t[name])}
results["sklearn_expected"] = {"threshold": None, **scores(
    y_test, month_test, w_test, p=proba["test"]["sklearn"], estimate=proba["test"]["sklearn"])}
naive_p = np.full(len(y_test), train_rate)
results["naive_constant"] = {"threshold": None, **scores(y_test, month_test, w_test, p=naive_p, estimate=naive_p)}
results["all_return"] = {"threshold": None, **scores(y_test, month_test, w_test, assign=np.zeros(len(y_test), bool))}
results["closest_party_rule"] = {"threshold": None, **scores(y_test, month_test, w_test, assign=rule_churn(te))}
comparison = pd.DataFrame(results).T
print(comparison.drop(columns=["false_pos", "false_neg"]).astype(float).round(4).to_string())

# Do the three implementations agree?
P = proba["test"]
coef = {
    "sklearn": np.r_[sk_model.intercept_, sk_model.coef_.ravel()],
    "manual_pytorch": np.r_[b_manual.numpy(), w_manual.numpy().ravel()],
    "nn_module": np.r_[nn_model.linear.bias.detach().numpy(), nn_model.linear.weight.detach().numpy().ravel()],
}
agreement = {
    f"{a}_vs_{b}": {
        "max_abs_dprob": float(np.max(np.abs(P[a] - P[b]))),
        "mean_abs_dprob": float(np.mean(np.abs(P[a] - P[b]))),
        "max_abs_dcoef": float(np.max(np.abs(coef[a] - coef[b]))),
        "same_assignment": float(np.mean((P[a] > best_t[a]) == (P[b] > best_t[b]))),
    }
    for a, b in (("sklearn", "manual_pytorch"), ("sklearn", "nn_module"), ("manual_pytorch", "nn_module"))
}
print(pd.DataFrame(agreement).T.to_string())

                    threshold    auc  logloss   brier  estimated_churn  actual_churn  error_pp  monthly_abs_error_pp  accuracy
sklearn                  0.32  0.914   0.2987  0.0902           0.2500        0.2488    0.1198                0.5029    0.8684
manual_pytorch           0.32  0.914   0.2987  0.0902           0.2500        0.2488    0.1198                0.5029    0.8684
nn_module                0.32  0.914   0.2987  0.0902           0.2500        0.2488    0.1198                0.5029    0.8684
sklearn_expected          NaN  0.914   0.2987  0.0902           0.2365        0.2488   -1.2324                1.5088       NaN
naive_constant            NaN  0.500   0.5614  0.1870           0.2366        0.2488   -1.2215                2.7918       NaN
all_return                NaN    NaN      NaN     NaN           0.0000        0.2488  -24.8802               24.9180    0.7512
closest_party_rule        NaN    NaN      NaN     NaN           0.3289        0.2488    8.0058                8

## How firm are the test numbers?

A bootstrap over test voters (1,000 resamples, drawn within each month so every month keeps its size) for AUC
and for the monthly error of the model and of the closest-party rule. With only four test months, the error is
an average of four numbers: the intervals show how much it moves with the voters drawn, not with the months.

In [16]:
p_test = proba["test"]["sklearn"]
assign_model = p_test > best_t["sklearn"]
assign_rule = rule_churn(te)
month_rows = [np.flatnonzero(month_test == s) for s in np.unique(month_test)]

boot = {"auc": [], "model": [], "rule": [], "diff": []}
for _ in range(N_BOOT):
    idx = np.concatenate([rng.choice(rows, len(rows)) for rows in month_rows])
    err_model = monthly_abs_error(y_test[idx], assign_model[idx], month_test[idx], w_test[idx])
    err_rule = monthly_abs_error(y_test[idx], assign_rule[idx], month_test[idx], w_test[idx])
    boot["auc"].append(roc_auc_score(y_test[idx], p_test[idx]))
    boot["model"].append(err_model)
    boot["rule"].append(err_rule)
    boot["diff"].append(err_rule - err_model)

ci = lambda v: [float(np.quantile(v, 0.025)), float(np.quantile(v, 0.975))]  # noqa: E731
test_ci = {
    "auc": results["sklearn"]["auc"], "auc_ci": ci(boot["auc"]),
    "model_error_pp": results["sklearn"]["monthly_abs_error_pp"], "model_error_ci": ci(boot["model"]),
    "rule_error_pp": results["closest_party_rule"]["monthly_abs_error_pp"], "rule_error_ci": ci(boot["rule"]),
    "rule_minus_model_ci": ci(boot["diff"]),
}
print(json.dumps(test_ci, indent=1))

{
 "auc": 0.9140319425334422,
 "auc_ci": [
  0.9034943275249885,
  0.9241895197732165
 ],
 "model_error_pp": 0.5028532124072588,
 "model_error_ci": [
  0.4160623633986741,
  1.855350833961346
 ],
 "rule_error_pp": 8.01910011408621,
 "rule_error_ci": [
  6.745325496749284,
  9.428836816619684
 ],
 "rule_minus_model_ci": [
  5.472067616395254,
  8.506888443568377
 ]
}


## Month by month on the test set

Four test months is a small sample; the per-month errors show how much the headline moves.

In [17]:
per_month = []
p_sk = proba["test"]["sklearn"]
for study in np.unique(month_test):
    m = month_test == study
    y_m, p_m, w_m = y_test[m], p_sk[m], w_test[m]
    per_month.append({
        "study": int(study), "rows": int(m.sum()), "actual_churn": float(np.average(y_m, weights=w_m)),
        "auc": roc_auc_score(y_m, p_m),
        "model": float(np.average(p_m > best_t["sklearn"], weights=w_m)),
        "model_expected": float(np.average(p_m, weights=w_m)),
        "rule": float(np.average(assign_rule[m], weights=w_m)), "naive": train_rate,
    })
print(pd.DataFrame(per_month).round(3).to_string(index=False))

 study  rows  actual_churn   auc  model  model_expected  rule  naive
  3562  1078         0.206 0.918  0.218           0.211 0.297  0.237
  3567  1023         0.262 0.925  0.261           0.243 0.345  0.237
  3571  1098         0.268 0.889  0.267           0.247 0.333  0.237
  3577   973         0.261 0.922  0.255           0.246 0.342  0.237


## Would fitting with the corrected weights change anything?

The models above are fitted unweighted: the correction's demographic axes (sex, age, education) are features, and
every 2023 PSOE voter has the same recall, so the weights mostly add variance (design effect about 2). Check: the
same scikit-learn model fitted with `sample_weight` (normalised to mean 1, so `C` means the same), its own
threshold chosen on validation, scored the same way.

In [ ]:
sk_weighted = LogisticRegression(C=best_C, max_iter=10_000, tol=1e-10).fit(
    X_train, y_train, sample_weight=w_train / w_train.mean())
weighted_fit_check = {}
for name, model in (("unweighted (used)", sk_model), ("weighted", sk_weighted)):
    pv, pt = model.predict_proba(X_val)[:, 1], model.predict_proba(X_test)[:, 1]
    t = float(THRESHOLDS[int(np.argmin([monthly_abs_error(y_val, pv > t, month_val, w_val) for t in THRESHOLDS]))])
    weighted_fit_check[name] = {
        "val_logloss_weighted": log_loss(y_val, pv, sample_weight=w_val), "test_auc": roc_auc_score(y_test, pt),
        "threshold": t,
        "val_monthly_abs_error_pp": monthly_abs_error(y_val, pv > t, month_val, w_val),
        "val_monthly_abs_error_expected_pp": monthly_abs_error(y_val, pv, month_val, w_val),
        "test_monthly_abs_error_pp": monthly_abs_error(y_test, pt > t, month_test, w_test),
        "test_monthly_abs_error_expected_pp": monthly_abs_error(y_test, pt, month_test, w_test),
    }
print(pd.DataFrame(weighted_fit_check).round(4).to_string())

## Allocating the real undecided

What the model is for: the 2023 PSOE voters who gave no intention, in every month, assigned with the validation
threshold. Retention among *all* 2023 PSOE voters then combines the declared (known) and the undecided
(allocated). Weighted with the corrected weight, as a published estimate would be. These numbers have no
ground truth to check against.

The undecided are not a random slice of the declared, so the last table compares them on the model's inputs:
the bigger the difference, the further the model is being stretched outside what it was trained on.

In [18]:
allocation_rows = raw.loc[undecided, ["study", "weight"]].assign(
    p=proba["undecided"]["sklearn"], rule=rule_churn(undecided).astype(int))
allocation_rows["assigned"] = (allocation_rows["p"] > best_t["sklearn"]).astype(int)
declared_rows = raw.loc[declared, ["study", "weight"]].assign(churn=eng.loc[declared, F.TARGET])


def wmean(values, weights) -> float:
    return float(np.average(values, weights=weights))


allocation = []
for study, und in allocation_rows.groupby("study"):
    dec = declared_rows[declared_rows["study"] == study]
    w_dec, w_und = dec["weight"].sum(), und["weight"].sum()
    stays_dec = (dec["weight"] * (1 - dec["churn"])).sum()

    def retention(und_returns: pd.Series) -> float:
        return float((stays_dec + (und["weight"] * und_returns).sum()) / (w_dec + w_und))

    allocation.append({
        "study": int(study), "undecided": len(und), "undecided_share": float(w_und / (w_dec + w_und)),
        "return_model": 1 - wmean(und["assigned"], und["weight"]),
        "return_model_expected": 1 - wmean(und["p"], und["weight"]),
        "return_rule": 1 - wmean(und["rule"], und["weight"]),
        "retention_declared_only": float(stays_dec / w_dec),
        "retention_model": retention(1 - und["assigned"]),
        "retention_rule": retention(1 - und["rule"]),
        "retention_all_return": retention(pd.Series(1, index=und.index)),
    })
print(pd.DataFrame(allocation).round(3).to_string(index=False))

profile_cols = ["rate_sanchez", "gap_best_rival", "rate_feijoo", "ideology", "probvoto", "eco_spain"]
undecided_profile = pd.DataFrame({
    "declared": eng.loc[declared, profile_cols].mean(),
    "undecided": eng.loc[undecided, profile_cols].mean(),
})
undecided_profile.loc["closest_party_psoe"] = [
    float((eng.loc[declared, "closest_group"] == "psoe").mean()),
    float((eng.loc[undecided, "closest_group"] == "psoe").mean())]
print("\n" + undecided_profile.round(2).to_string())

 study  undecided  undecided_share  return_model  return_model_expected  return_rule  retention_declared_only  retention_model  retention_rule  retention_all_return
  3505        150            0.129         0.547                  0.639        0.646                    0.819            0.784           0.796                 0.842
  3510        143            0.142         0.646                  0.685        0.643                    0.817            0.793           0.792                 0.843
  3514        148            0.141         0.464                  0.584        0.517                    0.794            0.748           0.755                 0.824
  3517        202            0.188         0.421                  0.545        0.587                    0.684            0.634           0.666                 0.743
  3524        178            0.152         0.417                  0.550        0.580                    0.798            0.741           0.765                 0.829
  3528    

## Save artifacts

Everything the dashboard needs; it never refits anything. `predictions.csv` holds the declared voters of the
validation and test months and every undecided voter, with all three models' probabilities.

In [19]:
joblib.dump(preprocessor, ARTIFACTS / "preprocessor.joblib")
joblib.dump(segmenter, ARTIFACTS / "segmenter.joblib")
joblib.dump(sk_model, ARTIFACTS / "sklearn_logreg.joblib")
torch.save({"w": w_manual, "b": b_manual}, ARTIFACTS / "manual_pytorch.pt")
torch.save(nn_model.state_dict(), ARTIFACTS / "nn_module.pt")

keep = ["study", "field_date", "weight", "destination", "EDAD", "CERCANIA"]
pred_rows = []
for split, m in (("val", va), ("test", te), ("undecided", undecided)):
    df = pd.concat([raw.loc[m, keep], eng.loc[m]], axis=1).assign(set=split)
    for name, p in proba[split].items():
        df[f"p_{name}"] = p
    pred_rows.append(df)
pd.concat(pred_rows).to_csv(ARTIFACTS / "predictions.csv", index=False)

segment_profile = (eng.loc[tr].groupby("segment")
                   .agg(size=(F.TARGET, "size"), churn=(F.TARGET, "mean"), **{c: (c, "mean") for c in F.SEGMENT_INPUTS}))
metrics = {
    "split": {s: {"studies": sorted(raw.loc[m, "study"].unique().tolist()), "rows": int(m.sum()),
                  "churn_rate": float(eng.loc[m, F.TARGET].mean())}
              for s, m in (("train", tr), ("val", va), ("test", te))},
    "undecided_rows": int(undecided.sum()),
    "selected_blocks": selected, "ablation": ablation, "min_logloss_gain": MIN_LOGLOSS_GAIN,
    "ablation_check": ablation_check,
    "segments": {"k": best_k, "silhouettes": silhouettes, "inputs": F.SEGMENT_INPUTS,
                 "profile": segment_profile.reset_index().to_dict(orient="records")},
    "hyperparameters": {"C": best_C, "C_val_logloss": c_scores, "lambda": lambda_reg,
                        "learning_rate": best_lr, "lr_train_objective": lr_objective,
                        "sklearn_train_objective": sk_objective, "gd_stable_lr_bound": float(2 / L_bound),
                        "epochs": EPOCHS},
    "thresholds": best_t, "threshold_check": threshold_check, "train_churn_rate": train_rate,
    "comparison_test": comparison.to_dict(orient="index"), "test_ci": test_ci,
    "agreement_test": agreement, "per_month_test": per_month,
    "allocation": allocation, "undecided_profile": undecided_profile.to_dict(orient="index"),
    "raking": json.loads((ROOT / "data" / "raking_diagnostics.json").read_text()),
    "weighted_fit_check": weighted_fit_check,
    "feature_names": feature_names,
    "coefficients": {k: v.tolist() for k, v in coef.items()},
    "loss_history": {"manual_pytorch": hist_manual, "nn_module": hist_nn},
}
(ARTIFACTS / "metrics.json").write_text(json.dumps(metrics, indent=1, default=float))
print("saved:", sorted(p.name for p in ARTIFACTS.iterdir()))

saved: ['manual_pytorch.pt', 'metrics.json', 'nn_module.pt', 'predictions.csv', 'preprocessor.joblib', 'segmenter.joblib', 'sklearn_logreg.joblib']
